# 01 · Synthetic data generation
**Student Performance & Placement Predictor**

No suitable public dataset contains all of the required fields (academic record, coding/aptitude scores, projects,
internships, hackathons, *and* placement outcomes), and real student records are personal data. This project therefore
uses a **synthetic dataset**.

> ⚠️ **Synthetic data disclaimer.** Every record is simulated. Relationships between features and targets are
> assumptions written into the generator, plus random noise. Nothing learned from this data is evidence about
> real students or real placement outcomes.

## Generative story
1. Each simulated student gets two hidden traits: **ability** and **diligence** (standard normal).
2. Observable features are noisy functions of those traits, so they correlate realistically
   (e.g. diligence → attendance, study hours; ability → CGPA, coding score).
3. Targets are computed from observable features plus noise:
   - **Academic performance index** ← CGPA, attendance, backlogs, study hours → binned into Poor / Average / Good / Excellent
   - **Placement score** ← coding, aptitude, communication, technical skills, CGPA, projects, certifications, internship,
     hackathons, backlogs → binned into Not Ready / Developing / Ready / Highly Ready
   - **Placement probability** = logistic transform of the placement score + noise, in 0–100 %
4. Data-quality problems are injected deliberately: ~2 % missing values in six columns, 15 duplicate rows and
   6 impossible values (e.g. CGPA 12.4, attendance 250 %).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from src.generate_data import generate_students
from src.utils import DATASET_PATH, RANDOM_SEED
print('Seed:', RANDOM_SEED)

Seed: 42


In [2]:
df = generate_students(n=1200, seed=RANDOM_SEED)
print('Shape:', df.shape)
df.head(10)

Shape: (1215, 20)


,student_id,age,gender,cgpa,attendance_pct,backlogs,coding_score,aptitude_score,communication_score,projects,certifications,internship,hackathon,extracurricular_score,technical_skills_score,programming_languages,study_hours_per_week,academic_performance,placement_readiness,placement_probability
0,STU00740,20,Female,4.87,72.0,3,30.0,34,55.0,0,0.0,No,No,79.0,22,1,7.7,Poor,Not Ready,2.1
1,STU00789,20,Male,5.82,80.3,1,43.0,48,48.0,0,NaN,No,No,62.0,28,4,12.3,Poor,Not Ready,5.6
2,STU00044,23,Male,7.56,75.8,0,68.0,66,73.0,1,0.0,No,Yes,53.0,44,3,13.9,Good,Developing,47.7
3,STU00156,20,Male,9.23,90.1,0,85.0,70,61.0,5,3.0,Yes,Yes,54.0,73,3,22.9,Excellent,Highly Ready,99.5
4,STU00495,20,Male,6.35,75.4,2,51.0,50,66.0,1,0.0,No,No,59.0,32,1,7.3,Poor,Not Ready,7.7
5,STU00971,22,Male,5.86,68.1,1,35.0,58,63.0,0,0.0,No,No,48.0,16,2,11.7,Poor,Not Ready,1.8
6,STU00271,21,Male,5.84,79.2,0,42.0,51,92.0,1,2.0,No,No,52.0,26,3,11.8,Poor,Not Ready,11.5
7,STU00591,20,Female,5.58,71.4,4,48.0,53,45.0,0,0.0,No,No,50.0,32,1,2.3,Poor,Not Ready,0.5
8,STU00124,22,Female,7.18,74.6,1,80.0,87,79.0,2,0.0,Yes,No,28.0,56,5,10.3,Average,Highly Ready,93.0
9,STU00114,21,Male,6.31,78.6,0,36.0,38,82.0,1,3.0,Yes,No,62.0,25,2,19.7,Average,Developing,16.3


## Target distributions

In [3]:
for t in ['academic_performance', 'placement_readiness']:
    print(df[t].value_counts().to_string(), '\n')
df['placement_probability'].describe().round(2)

academic_performance
Average      371
Poor         340
Good         323
Excellent    181 

placement_readiness
Not Ready       422
Highly Ready    378
Developing      230
Ready           185 



count    1215.00
mean       47.73
std        37.30
min         0.50
25%         9.20
50%        42.60
75%        89.35
max        99.50
Name: placement_probability, dtype: float64

## Injected data-quality issues (to be handled by the preprocessing pipeline)

In [4]:
print('Duplicate rows :', df.duplicated().sum())
print('Missing values :'); print(df.isna().sum()[df.isna().sum() > 0].to_string())
print('\nImpossible values:')
print(df.loc[(df.cgpa > 10) | (df.cgpa < 0) | (df.attendance_pct > 100) | (df.study_hours_per_week > 100) | (df.coding_score < 0),
             ['student_id', 'cgpa', 'attendance_pct', 'coding_score', 'study_hours_per_week']])

Duplicate rows : 15
Missing values :
attendance_pct           26
communication_score      24
certifications           24
internship               24
extracurricular_score    25
study_hours_per_week     25

Impossible values:
    student_id   cgpa  attendance_pct  coding_score  study_hours_per_week
178   STU00426  -1.00            90.6          66.0                  22.2
288   STU01115  12.40            86.1          87.0                  20.6
702   STU00457   7.45           250.0          92.0                  16.6
788   STU00237   5.80             NaN          18.0                 160.0
833   STU00783   7.87            76.9          -8.0                  12.5
853   STU00471   7.60           134.0          60.0                   9.8


## Sanity check: do the designed relationships show up?

In [5]:
df.groupby('placement_readiness')[['cgpa','coding_score','projects','placement_probability']].mean().round(2).loc[['Not Ready','Developing','Ready','Highly Ready']]

,cgpa,coding_score,projects,placement_probability
placement_readiness,,,,
Not Ready,6.06,42.45,0.63,6.35
Developing,6.87,56.23,1.35,33.48
Ready,7.17,62.35,1.91,66.10
Highly Ready,7.88,71.97,3.33,93.60


## Save

In [6]:
DATASET_PATH.parent.mkdir(exist_ok=True)
df.to_csv(DATASET_PATH, index=False)
print(f'Saved {len(df)} rows to {DATASET_PATH.relative_to(ROOT)}')

Saved 1215 rows to data/student_dataset.csv
